# 03 — Gold Aggregation + Dimensional Model

Silver (route-day) → **Gold** (route-month) + a **flat star schema** for a Route
Profitability dashboard sliceable by date, region, BU, area.

`sql/dimensional_model_ddl.sql` is **generated at the end of this notebook from the
real Delta schemas written here**, so the DDL is the literal CREATE statements for
what actually ran — it cannot drift from the code.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("../src"))
from pyspark.sql import functions as F
from spark_session import build_spark, delta_path
from metrics import compute_underperformance_threshold

spark = build_spark("03_gold_dimensional")
SILVER   = delta_path("silver", "route_day_profitability", spark)
GOLD     = delta_path("gold",   "route_month_profitability", spark)
DIM_ROUTE = delta_path("gold",  "dim_route", spark)
DIM_DATE  = delta_path("gold",  "dim_date", spark)
silver = spark.read.format("delta").load(SILVER)

# Same data-driven threshold Silver used (exact P10, rounded) — recomputed from the
# same column so Gold's month-level flag stays consistent with Silver's day flag.
THRESHOLD = compute_underperformance_threshold(silver, "gross_margin_pct", q=0.10)
print("Spark", spark.version, "| Silver rows:", silver.count(),
      "| underperformance threshold:", THRESHOLD, "%")

Spark 4.0.3 | Silver rows: 12000 | underperformance threshold: 10.0 %


## 1. Gold grain: **route × month** — justified against alternatives

Bronze/Silver hold **12,000 route-days** = 120 routes × ~100 days each over the
3-year window (2022-01 .. 2024-12, 36 months). Three candidate Gold grains:

| grain | row count | verdict |
|---|---|---|
| **route-day** (= Silver) | 12,000 | Too fine for a dashboard. Every BI slice re-aggregates 12k rows on the fly; and a single day is noisy (one heavy haul swings margin). This is the *analysis* grain, not the *serving* grain. |
| **route-month** (chosen) | ≤ 120 × 36 = **4,320** (fewer — not every route runs every month) | A month smooths daily noise, aligns with how ops reviews P&L (monthly close), and keeps **route-level accountability** (you can still name the bad route). Small enough that dashboard slices by region/BU/area are instant. |
| **route only** | 120 | Throws away the time axis — can't answer the assignment's "improving or deteriorating across the 3-year window". Too coarse. |

**Route-month** is the sweet spot: ~3x smaller than Silver, monthly trend
preserved, route identity preserved. Margin is **revenue-weighted** within the
month (see next cell), never a mean of daily percentages.

In [2]:
def wdiv(num, den):
    return F.when((F.col(den).isNull()) | (F.col(den) == 0), None).otherwise(F.col(num) / F.col(den))

# Sum the additive base measures over the month; geography is constant per route
# so we carry it through with first().
gold = (
    silver.groupBy("route_id", "year", "month", "quarter", "region", "bu", "area")
    .agg(
        F.count(F.lit(1)).alias("n_route_days"),
        F.sum("total_revenue").alias("total_revenue"),
        F.sum("total_cost").alias("total_cost"),
        F.sum("net_revenue").alias("net_revenue"),
        F.sum("gross_profit").alias("gross_profit"),
        F.sum("disposal_cost").alias("disposal_cost"),
        F.sum("fuel_cost").alias("fuel_cost"),
        F.sum("labour_cost").alias("labour_cost"),
        F.sum("maintenance_cost").alias("maintenance_cost"),
        F.sum("admin_cost").alias("admin_cost"),
        F.sum("total_tonnes").alias("total_tonnes"),
        F.sum("total_yards").alias("total_yards"),
        F.sum("total_distance_km").alias("total_distance_km"),
        F.sum("total_fuel_used_litres").alias("total_fuel_used_litres"),
        F.sum("total_labour_hours").alias("total_labour_hours"),
        F.sum("completed_stops").alias("completed_stops"),
        F.sum("total_stops").alias("total_stops"),
        F.sum("missed_stops").alias("missed_stops"),
        F.sum("incident_flag").alias("incident_days"),
        F.sum(F.col("underperforming_flag").cast("int")).alias("underperforming_days"),
    )
)

# Weighted / ratio measures recomputed at month grain from the summed bases.
gold = (
    gold
    .withColumn("weighted_gross_margin_pct", wdiv("gross_profit", "total_revenue") * 100)
    .withColumn("revenue_per_completed_stop", wdiv("total_revenue", "completed_stops"))
    .withColumn("cost_per_completed_stop",    wdiv("total_cost", "completed_stops"))
    .withColumn("profit_per_completed_stop",  wdiv("gross_profit", "completed_stops"))
    .withColumn("revenue_per_tonne",      wdiv("total_revenue", "total_tonnes"))
    .withColumn("cost_per_tonne",         wdiv("total_cost", "total_tonnes"))
    .withColumn("profit_per_tonne",       wdiv("gross_profit", "total_tonnes"))
    .withColumn("disposal_cost_per_tonne", wdiv("disposal_cost", "total_tonnes"))
    .withColumn("fuel_litres_per_100km",
                F.when((F.col("total_distance_km").isNull()) | (F.col("total_distance_km") == 0), None)
                 .otherwise(F.col("total_fuel_used_litres") * 100 / F.col("total_distance_km")))
    .withColumn("stops_per_labour_hour",  wdiv("completed_stops", "total_labour_hours"))
    .withColumn("completion_rate",        wdiv("completed_stops", "total_stops"))
    .withColumn("missed_stop_rate",       wdiv("missed_stops", "total_stops"))
    .withColumn("disposal_cost_share",    wdiv("disposal_cost", "total_cost"))
    .withColumn("fuel_cost_share",        wdiv("fuel_cost", "total_cost"))
    .withColumn("labour_cost_share",      wdiv("labour_cost", "total_cost"))
    .withColumn("maintenance_cost_share", wdiv("maintenance_cost", "total_cost"))
    .withColumn("admin_cost_share",       wdiv("admin_cost", "total_cost"))
    .withColumn("pct_underperforming_days", wdiv("underperforming_days", "n_route_days"))
    # Month-level underperformance: same data-driven floor as Silver (THRESHOLD),
    # applied to the revenue-weighted monthly margin.
    .withColumn("month_underperforming_flag", F.col("weighted_gross_margin_pct") < F.lit(THRESHOLD))
)

print("Gold grain check — rows:", gold.count(),
      "| distinct (route,year,month):", gold.select("route_id","year","month").distinct().count())
gold.select("route_id","year","month","n_route_days","weighted_gross_margin_pct",
            "disposal_cost_per_tonne","month_underperforming_flag").show(5, truncate=False)

Gold grain check — rows: 4035 | distinct (route,year,month): 4035


+--------+----+-----+------------+-------------------------+-----------------------+--------------------------+
|route_id|year|month|n_route_days|weighted_gross_margin_pct|disposal_cost_per_tonne|month_underperforming_flag|
+--------+----+-----+------------+-------------------------+-----------------------+--------------------------+
|RTE-0020|2023|5    |4           |14.971109435769867       |80.11314011553903      |false                     |
|RTE-0025|2023|10   |4           |58.50380552138884        |72.692130126691        |false                     |
|RTE-0040|2023|5    |8           |35.23275864776664        |72.85681259581924      |false                     |
|RTE-0061|2023|10   |3           |-11.742712768903468      |76.65823255813953      |true                      |
|RTE-0061|2023|3    |5           |-9.757310452990527       |78.63358304403435      |true                      |
+--------+----+-----+------------+-------------------------+-----------------------+--------------------

## 2. Write Gold, partitioned by `region`, `bu`, `area`

These three are the dashboard's primary slice keys. Partitioning on the
geography hierarchy means a "show me Ontario / GTA BU / Brampton" filter prunes
straight to the relevant files. The fact is small (~thousands of rows) and
**append-mostly** (a new month lands monthly), so partition pruning + periodic
compaction is all the physical tuning it needs.

In [3]:
(gold.write.format("delta").mode("overwrite").option("overwriteSchema","true")
   .partitionBy("region","bu","area").save(GOLD))
g = spark.read.format("delta").load(GOLD)
print("GOLD route_month_profitability rows:", g.count())

GOLD route_month_profitability rows: 4035


## 3. Dimensional model — **flat star** (chosen), and why *not* snowflake

```
            dim_date (year, month grain)
                 |
                 |  (year, month)
                 v
   dim_route --< fact_route_month_profitability
   (route_id)     (route_id, year, month)
```

**Chosen: a flat star.** `region / bu / area` are **denormalized directly onto
`dim_route`**, with **no surrogate keys** — the natural keys (`route_id`, and the
`(year, month)` pair) are stable, unique, and human-readable.

**Why not a snowflake (`dim_region → dim_business_unit → dim_area → dim_route` with
surrogate keys):**

- The assignment describes a **static** geography/operational hierarchy and a
  **single, append-mostly fact table**. There is **no slowly-changing dimension
  requirement** — a route doesn't migrate BUs mid-history in this dataset (1 BU per
  route, verified below). Snowflaking + SCD2 surrogate-key machinery would add 3
  extra joins and a key-management burden to model change that **cannot happen here**.
- Star geometry means every dashboard slice is **one join** (`fact → dim_route`)
  instead of a four-table walk. At route-month grain that keeps slices instant.
- Denormalized geography on `dim_route` is tiny (120 rows) — the storage "cost" of
  denormalization is negligible, and Delta's columnar + ZORDER handles the slice
  performance the snowflake would otherwise be justifying.

`dim_route` carries the geo hierarchy; `dim_date` is at **month grain** (one row per
year-month, with quarter) to match the Gold fact. Dimensions are built from the
**real distinct values in Silver**, not hardcoded.

In [4]:
# Confirm the modeling assumption: each route maps to exactly one region/bu/area.
viol = (silver.groupBy("route_id")
        .agg(F.countDistinct("region").alias("r"), F.countDistinct("bu").alias("b"),
             F.countDistinct("area").alias("a"))
        .filter("r>1 OR b>1 OR a>1").count())
print("routes with >1 region/bu/area (must be 0 for flat star):", viol)

# dim_route: natural key route_id + denormalized geography. Built from real data.
dim_route = (silver.select("route_id","region","bu","area").distinct()
             .orderBy("route_id"))

# dim_date at MONTH grain: one row per (year, month), natural key date_month_key.
dim_date = (silver.select("year","month","quarter").distinct()
            .withColumn("date_month_key", F.col("year")*100 + F.col("month"))
            .withColumn("month_start_date",
                        F.to_date(F.concat_ws("-", F.col("year"), F.col("month"), F.lit(1))))
            .select("date_month_key","year","month","quarter","month_start_date")
            .orderBy("date_month_key"))

dim_route.write.format("delta").mode("overwrite").option("overwriteSchema","true").save(DIM_ROUTE)
dim_date.write.format("delta").mode("overwrite").option("overwriteSchema","true").save(DIM_DATE)

print("dim_route rows:", spark.read.format("delta").load(DIM_ROUTE).count())
print("dim_date  rows:", spark.read.format("delta").load(DIM_DATE).count())

routes with >1 region/bu/area (must be 0 for flat star): 0


dim_route rows: 120


dim_date  rows: 36


## 4. Samples — dimensions and Gold fact

In [5]:
print("=== dim_route (sample) ===")
spark.read.format("delta").load(DIM_ROUTE).show(5, truncate=False)
print("=== dim_date (sample) ===")
spark.read.format("delta").load(DIM_DATE).orderBy("date_month_key").show(5, truncate=False)
print("=== fact route_month_profitability (sample) ===")
(spark.read.format("delta").load(GOLD)
   .select("route_id","year","month","quarter","region","bu","area",
           "n_route_days","total_revenue","weighted_gross_margin_pct",
           "disposal_cost_per_tonne","month_underperforming_flag")
   .orderBy("route_id","year","month").show(8, truncate=False))

=== dim_route (sample) ===


+--------+--------+-----------+-----------+
|route_id|region  |bu         |area       |
+--------+--------+-----------+-----------+
|RTE-0001|Prairies|Prairies BU|Winnipeg   |
|RTE-0002|Atlantic|Atlantic BU|Fredericton|
|RTE-0003|Quebec  |Quebec BU  |Laval      |
|RTE-0004|Quebec  |Quebec BU  |Quebec City|
|RTE-0005|BC      |BC BU      |Vancouver  |
+--------+--------+-----------+-----------+
only showing top 5 rows
=== dim_date (sample) ===


+--------------+----+-----+-------+----------------+
|date_month_key|year|month|quarter|month_start_date|
+--------------+----+-----+-------+----------------+
|202201        |2022|1    |Q1     |2022-01-01      |
|202202        |2022|2    |Q1     |2022-02-01      |
|202203        |2022|3    |Q1     |2022-03-01      |
|202204        |2022|4    |Q2     |2022-04-01      |
|202205        |2022|5    |Q2     |2022-05-01      |
+--------------+----+-----+-------+----------------+
only showing top 5 rows
=== fact route_month_profitability (sample) ===


+--------+----+-----+-------+--------+-----------+--------+------------+------------------+-------------------------+-----------------------+--------------------------+
|route_id|year|month|quarter|region  |bu         |area    |n_route_days|total_revenue     |weighted_gross_margin_pct|disposal_cost_per_tonne|month_underperforming_flag|
+--------+----+-----+-------+--------+-----------+--------+------------+------------------+-------------------------+-----------------------+--------------------------+
|RTE-0001|2022|1    |Q1     |Prairies|Prairies BU|Winnipeg|3           |23141.33          |40.94418946534188        |74.39114361488386      |false                     |
|RTE-0001|2022|2    |Q1     |Prairies|Prairies BU|Winnipeg|2           |15690.06          |39.86128797467951        |79.6650581539223       |false                     |
|RTE-0001|2022|3    |Q1     |Prairies|Prairies BU|Winnipeg|3           |20344.800000000003|36.46165113444221        |77.001100668868        |false         

## 5. Generate `sql/dimensional_model_ddl.sql` from the real schemas

Emit literal `CREATE TABLE` statements straight off the written Delta tables, so
the SQL file is provably the same schema as the code above (column names + types
copied, not paraphrased).

In [6]:
def ddl_for(path, table, partitions, pk_comment):
    sch = spark.read.format("delta").load(path).schema
    lines = [f"    {f.name} {f.dataType.simpleString().upper()}" for f in sch.fields]
    body = ",\n".join(lines)
    part = f"\nPARTITIONED BY ({', '.join(partitions)})" if partitions else ""
    return (f"-- {pk_comment}\nCREATE TABLE {table} (\n{body}\n)\nUSING DELTA{part};\n")

header = """-- =====================================================================
-- GFL Route Profitability — Dimensional Model DDL (flat star schema)
-- AUTO-GENERATED from the live Delta schemas in 03_gold_and_dimensional_model.ipynb.
-- Grain of fact: one row per route per month. No surrogate keys (natural keys are
-- stable + unique). Geography denormalized onto dim_route (no snowflake / SCD2).
--
--   dim_date (date_month_key)  --+
--                                 >--  fact_route_month_profitability
--   dim_route (route_id)       --+      (route_id, year, month)
--
-- Relationships (enforced in BI tool, not by Delta):
--   fact.route_id              -> dim_route.route_id
--   (fact.year, fact.month)    -> dim_date.(year, month)   [date_month_key = year*100+month]
-- =====================================================================

"""

ddl = header
ddl += ddl_for(DIM_DATE,  "dim_date",  None,
               "DIMENSION: calendar at month grain. PK: date_month_key (= year*100 + month).")
ddl += "\n" + ddl_for(DIM_ROUTE, "dim_route", None,
               "DIMENSION: route with denormalized geography. PK: route_id.")
ddl += "\n" + ddl_for(GOLD, "fact_route_month_profitability", ["region","bu","area"],
               "FACT: route-month profitability. PK: (route_id, year, month). Partitioned by geography.")

out_sql = os.path.abspath("../sql/dimensional_model_ddl.sql")
with open(out_sql, "w") as f:
    f.write(ddl)
print("wrote", out_sql)
print("\n================ dimensional_model_ddl.sql ================\n")
print(ddl)

wrote C:\Users\ramam\Downloads\gfl-route-profitability\gfl-route-profitability\sql\dimensional_model_ddl.sql

================ dimensional_model_ddl.sql ================

-- =====================================================================
-- GFL Route Profitability — Dimensional Model DDL (flat star schema)
-- AUTO-GENERATED from the live Delta schemas in 03_gold_and_dimensional_model.ipynb.
-- Grain of fact: one row per route per month. No surrogate keys (natural keys are
-- stable + unique). Geography denormalized onto dim_route (no snowflake / SCD2).
--
--   dim_date (date_month_key)  --+
--                                 >--  fact_route_month_profitability
--   dim_route (route_id)       --+      (route_id, year, month)
--
-- Relationships (enforced in BI tool, not by Delta):
--   fact.route_id              -> dim_route.route_id
--   (fact.year, fact.month)    -> dim_date.(year, month)   [date_month_key = year*100+month]
-- ====================================================

## 6. Delta features that shaped this design (tied to this use case)

- **Partitioning** — Gold partitioned by `region/bu/area` (the dashboard's primary
  slice keys) so leadership filters prune files instead of scanning the whole fact.
  Bronze/Silver partitioned by time (`year`+`month`/`region`) matching their
  ingest/analysis access patterns.
- **OPTIMIZE + ZORDER** (Silver, demonstrated in 02) — co-locates `bu/area/route_id`
  within partitions so the finer "drill into one route" queries hit few files.
  Confirmed working on OSS Delta 4.0, not a Databricks-only no-op.
- **MERGE** (Bronze, demonstrated in 01) — the reason we *don't* need SCD2 here:
  late disposal invoices / billing restatements are handled as **upserts on
  `route_date_key`** at Bronze, so corrections flow forward on rebuild. The
  dimensional layer stays simple because the mutation is absorbed upstream.
- **Time travel** — Gold/Silver are full `overwrite` rebuilds, so each run is a new
  version. `DESCRIBE HISTORY` + `VERSION AS OF` let us reproduce any past dashboard
  state or diff month-over-month restatements for audit — valuable when a disposal
  invoice restates a closed month.
- **Schema evolution** — `overwriteSchema=true` lets new derived metrics be added to
  Silver/Gold without a manual migration; the enforced Bronze schema still guards
  the raw contract (FAILFAST) so evolution is deliberate, not accidental.

Star (not snowflake) is itself a consequence: Delta's columnar storage + ZORDER
deliver the slice performance a snowflake would try to buy with normalization, so
we keep the simpler one-join model and avoid SCD2 we have no requirement for.

In [7]:
spark.stop()
print("Spark stopped.")

Spark stopped.
